In [ ]:
import numpy as np
from scipy.special import comb
from scipy.integrate import *

import matplotlib.pyplot as plt

from electron_integrals import *
from DirectCI import DirectCI

Define functions

In [ ]:
np.array([1,2,3])*np.array([2,3,4])

In [ ]:
from itertools import combinations

In [ ]:
I_a = DirectCI.get_slater_dets(3,1)
print(I_a)
np.mgrid(I_a, I_a)

In [ ]:
A = np.array([1,1,1,1,1])
A[3:1]

In [ ]:
def single_exc(p,q,n):
        n_pq = np.copy(n)
        n_pq[q] = 0

        if n_pq[p]==1:
            return 0, 0

        n_pq[p] = 1
        gamma = (-1)**(np.sum(n[:q])+np.sum(n_pq[:p]))

        return gamma, n_pq

single_exc(1,2,[1,0,0,1])

In [ ]:
for i in I_a:
    for j in I_a:
        print(np.concatenate([i,j]))

In [ ]:
DirectCI.get_slater_dets(num_orbitals, num_electrons).shape

In [ ]:
D, d = get_RDMs(num_orbitals, num_electrons, C_init)
dCI = DirectCI(h,g,num_orbitals,num_electrons,0)
D_2 = dCI.get_D(C_init.reshape(1,-1))

np.testing.assert_almost_equal(D,D_2)


In [ ]:
def get_D(num_orbitals, C, slater_dets):
    C_conj = C.conj()
    D = np.zeros((num_orbitals,)*2 , dtype=np.cdouble)

    for n, det_n in enumerate(slater_dets):
        for m, det_m in enumerate(slater_dets):
            num_differences = np.sum(np.abs(det_n-det_m))
            match(num_differences):
                case 0:
                    for p, n_p in enumerate(det_n):
                        D[p,p] += n_p*C_conj[m]*C[n]
                case 2:
                    p = np.flatnonzero(np.asarray((det_m-det_n)==1))[0]
                    q = np.flatnonzero(np.asarray((det_n-det_m)==1))[0]
                    gamma = (-1)**(np.sum(det_n[:q])+np.sum(det_m[:p]))
                    D[p,q] += gamma*C_conj[m]*C[n]


    return D

def get_d(num_orbitals, C, slater_dets_sigma, slater_dets_tau):
    C_conj = C.conj()

    D = np.zeros((num_orbitals,)*2 , dtype=np.cdouble)
    d = np.zeros((num_orbitals,)*4, dtype=np.cdouble)

    for n, det_n in enumerate(slater_dets):
        for m, det_m in enumerate(slater_dets):
            num_differences = np.sum(np.abs(det_n-det_m))
            match(num_differences):
                case 0:
                    for p, n_p in enumerate(det_n):
                        D[p,p] += n_p*C_conj[m]*C[n]
                        for r, n_r in enumerate(det_n):
                            d[p,r,p,r] += n_p*n_r*C_conj[m]*C[n]
                            d[p,r,r,p] -= n_p*n_r*C_conj[m]*C[n] # Note the sign                 
                case 2:
                    p = np.flatnonzero(np.asarray((det_m-det_n)==1))[0]
                    q = np.flatnonzero(np.asarray((det_n-det_m)==1))[0]
                    gamma = (-1)**(np.sum(det_n[:q])+np.sum(det_m[:p]))
                    D[p,q] += gamma*C_conj[m]*C[n]

                    for r, n_r in enumerate(det_n):
                        val = gamma*n_r*C_conj[m]*C[n]
                        d[p,r,q,r] += val
                        d[r,p,q,r] -= val # Note the sign
                        d[p,r,r,q] -= val # Note the sign
                        d[r,p,r,q] += val

                case 4:
                    p,q = np.flatnonzero(np.asarray((det_m-det_n)==1))
                    r,s = np.flatnonzero(np.asarray((det_n-det_m)==1))
                    
                    gamma = np.sum(det_n[:r])
                    gamma += np.sum(det_n[:s])-1 # -1 since r<s 
                    gamma += np.sum(det_n[:q])-int(s<q)-int(r<q)
                    gamma += np.sum(det_n[:p])-int(s<p)-int(r<p) # No additional term since p<q

                    val = C_conj[m]*C[n]*(-1)**gamma
                    d[p,q,r,s] += val
                    d[q,p,r,s] -= val # Note the sign
                    d[p,q,s,r] -= val # Note the sign
                    d[q,p,s,r] += val

    return D, d

In [ ]:
# Function to get reduced density matrices

def get_RDMs(num_orbitals, num_electrons, C):
    # Should probably get these as an argument instead.
    slater_dets = DirectCI.get_slater_dets(num_orbitals, num_electrons)
    C_conj = C.conj()

    D = np.zeros((num_orbitals,)*2 , dtype=np.cdouble)
    d = np.zeros((num_orbitals,)*4, dtype=np.cdouble)

    for n, det_n in enumerate(slater_dets):
        for m, det_m in enumerate(slater_dets):
            num_differences = np.sum(np.abs(det_n-det_m))
            match(num_differences):
                case 0:
                    for p, n_p in enumerate(det_n):
                        D[p,p] += n_p*C_conj[m]*C[n]
                        for r, n_r in enumerate(det_n):
                            d[p,r,p,r] += n_p*n_r*C_conj[m]*C[n]
                            d[p,r,r,p] -= n_p*n_r*C_conj[m]*C[n] # Note the sign                 
                case 2:
                    p = np.flatnonzero(np.asarray((det_m-det_n)==1))[0]
                    q = np.flatnonzero(np.asarray((det_n-det_m)==1))[0]
                    gamma = (-1)**(np.sum(det_n[:q])+np.sum(det_m[:p]))
                    D[p,q] += gamma*C_conj[m]*C[n]

                    for r, n_r in enumerate(det_n):
                        val = gamma*n_r*C_conj[m]*C[n]
                        d[p,r,q,r] += val
                        d[r,p,q,r] -= val # Note the sign
                        d[p,r,r,q] -= val # Note the sign
                        d[r,p,r,q] += val

                case 4:
                    p,q = np.flatnonzero(np.asarray((det_m-det_n)==1))
                    r,s = np.flatnonzero(np.asarray((det_n-det_m)==1))
                    
                    gamma = np.sum(det_n[:r])
                    gamma += np.sum(det_n[:s])-1 # -1 since r<s 
                    gamma += np.sum(det_n[:q])-int(s<q)-int(r<q)
                    gamma += np.sum(det_n[:p])-int(s<p)-int(r<p) # No additional term since p<q

                    val = C_conj[m]*C[n]*(-1)**gamma
                    d[p,q,r,s] += val
                    d[q,p,r,s] -= val # Note the sign
                    d[p,q,s,r] -= val # Note the sign
                    d[q,p,s,r] += val

    return D, d


# Helper functions to put the problem on a form handled by standard SciPy ODE solvers 
#@numba.njit
def Cb_to_y(C, b):
    return np.concatenate([C.flatten(), b.flatten()])

#@numba.njit
def y_to_Cb(y, num_mctdhf_orbitals, num_spin_orbitals, num_slater_dets):

    C, b = np.split(y, [num_slater_dets])
    b = b.reshape((num_spin_orbitals, num_mctdhf_orbitals))

    return C, b

def gram_schmidt(b):
    b = np.divide(b, np.sqrt(np.einsum('ij, ij -> j', b.conj(), b)))
    for i in range(b.shape[1]):

        orto_adjustment = 0
        for j in range(0, i):
            orto_adjustment += np.dot(b[:, i], b[:, j])*b[:, j]

        b[:,i] -= orto_adjustment

    return b

Define number of electrons and orbitals

In [ ]:
# Number of orbitals (without spin)
num_orbitals = 10
# Number of electrons
num_electrons = 2
#Include spin?
include_spin = False
spin_factor = 1+int(include_spin)

num_spin_orbitals = spin_factor*num_orbitals

Calculate electron integrals

In [ ]:
x_max = 10
num_points = 1000

x = np.linspace(-x_max,x_max,num_points)

#pot = GaussianWell(w=100, a=1, center=0)
pot = HOPotential()

spf, h = get_spf_and_diag_h(num_orbitals, x, pot)

if include_spin:
    #Add spin
    h = np.kron(h, np.eye(2,2))

g = coulomb_interaction_matrix_elements(spf, spf, x, x, kappa = 1, a = 0.01)

if include_spin:
    #Add spin
    g = np.kron(g, np.einsum("pr,qs->pqrs",np.eye(2,2), np.eye(2,2)))

print('Sanity test, due to symmetry in g this should be zero:')
print(-g[2,1,1,0]+g[2,0,1,1]+g[1,1,2,0]-g[1,0,2,1])

Solve using Slater-Condon 

In [ ]:
H = SlaterCondonHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E_CI, C_CI = np.linalg.eigh(H)
print(E_CI)
#print(C_CI[:,0].T@H@C_CI[:,0])

#print(C_CI.conj().T@C_CI)

D, d = get_RDMs(num_spin_orbitals, num_electrons, C_CI[:,0])
E_rdm = np.einsum('pq, pq', D, h) + 0.5 * np.einsum('pqrs, pqrs', g, d)
print(E_rdm)

E_exact = E_CI[0]

np.testing.assert_almost_equal(E_CI[0], E_rdm)


# Orbital equations

Define number of MCTDHF orbitals $(\{\ket{\phi_n(t)}\})$, initial b's (basis reduction coefficients),

$\ket{\phi_n(t)} = \sum_{k=1}^{N_b}b_{kn}(t)\ket{\psi_k}$,

and initial C's.


In [ ]:
num_mctdhf_orbitals = 10
num_mctdhf_slater_dets = int(comb(num_mctdhf_orbitals, num_electrons))

# Distributed vales
#b_init = np.zeros((num_spin_orbitals, num_mctdhf_orbitals))
#a = int(num_spin_orbitals/num_mctdhf_orbitals)
#for i in range(num_mctdhf_orbitals):
#    b_init[a*i:a*(i+1), i] = np.ones(a)

# Identity
#b_init=np.eye(num_spin_orbitals, num_mctdhf_orbitals)

# Generate random orthonormal vectors
rng = np.random.default_rng()
r = rng.random((num_spin_orbitals, num_mctdhf_orbitals))
u, _, vh = np.linalg.svd(r, full_matrices=False)
b_init = (u@vh).astype(np.cdouble)

#print(b_init)

# Ones as initial C
#C_init = np.ones((num_mctdhf_slater_dets), dtype=np.cdouble)

# Random initial C
C_init = rng.random(num_mctdhf_slater_dets).astype(np.cdouble)

#Normalize
C_init = np.divide(C_init, np.sqrt(C_init.T.conj()@C_init))

#C_init = np.zeros(num_mctdhf_slater_dets).astype(np.cdouble)
#C_init[0] = 1

#print(C_init)

Define callable class for the differential equations

In [ ]:
class MCDTHF_imaginary_time:
    def __init__(self, h, g, num_mctdhf_orbitals, num_spin_orbitals, num_electrons):
        self.h = h
        self.g = g
        self.num_mctdhf_orbitals = num_mctdhf_orbitals
        self.num_spin_orbitals = num_spin_orbitals
        self.num_electrons = num_electrons
        self.num_slater_dets = int(comb(num_mctdhf_orbitals, num_electrons))
        self.I = np.eye(self.num_slater_dets)
    
    def __call__(self, t, y):
        
        C, b = y_to_Cb(y, self.num_mctdhf_orbitals, self.num_spin_orbitals, self.num_slater_dets)
        bc = b.conj()

        ## These two implementations are identical. Not sure which is better.
        h_1 = np.einsum('jm, ij -> im', b, self.h)
        h_2 = np.einsum('in, im -> nm', bc, h_1)
        h_3 = np.einsum('in, nm -> im', b, h_2)
        # h_1 = self.h@b
        # h_2 = bc.T@h_1
        # h_3 = b@h_2

        g_2 = np.einsum('jq, ls, ijkl -> iqks', bc, b, self.g)
        g_3 = np.einsum('kr, iqks -> iqrs', b, g_2)
        g_4 = np.einsum('ip, iqrs -> pqrs', bc, g_3)
        g_5 = np.einsum('ip, pqrs -> iqrs', b, g_4)

        
        D, d = get_RDMs(self.num_mctdhf_orbitals, self.num_electrons, C)
        D_inv = np.linalg.pinv(D)

        b_dot = -(h_1 - h_3 + np.einsum('np, pqrs, iqrs -> in', D_inv, d, g_3-g_5))


        H = SlaterCondonHamiltonian(self.num_mctdhf_orbitals, self.num_electrons, h_2, g_4).get_hamiltonian()

        ## From Beck paper, replacing HC with (H-IE)C should keep the wave function normalized (should be able to remove renormalization further down)
        E = C.conj().T@H@C/(C.conj().T@C)
        C_dot = - (H-self.I*E)@C 
    
        ## Regular way
        #C_dot = -H@C


        return Cb_to_y(C_dot, b_dot)

Solve using ODE solver

In [ ]:
#mctdhf_fun = MCDTHF(h, g, num_mctdhf_orbitals, num_spin_orbitals, num_electrons)
mctdhf_fun = MCDTHF_imaginary_time(h, g, num_mctdhf_orbitals, num_spin_orbitals, num_electrons)

y_init = Cb_to_y(C_init, b_init)

Cs = [C_init]
bs = [b_init]

h_1 = np.einsum('jm, ij -> im', b_init, h)
h_2 = np.einsum('in, im -> nm', b_init.conj(), h_1)
g_2 = np.einsum('jq, ls, ijkl -> iqks', b_init.conj(), b_init, g)
g_3 = np.einsum('kr, iqks -> iqrs', b_init, g_2)
g_4 = np.einsum('ip, iqrs -> pqrs', b_init.conj(), g_3)


H = SlaterCondonHamiltonian(num_mctdhf_orbitals, num_electrons, h_2, g_4).get_hamiltonian()
Es = [C_init.conj().T@H@C_init]

t_init = 0
t_final = 5.0
solver = DOP853(mctdhf_fun, t_init, y_init, t_final, max_step = 1e-2)
#solver = RK45(mctdhf_fun, t_init, y_init, t_final, max_step = 5e-3)
#solver = RK23(mctdhf_fun, t_init, y_init, t_final, max_step = 0.001)

while solver.status == 'running':
    msg = solver.step()

    C, b = y_to_Cb(solver.y, num_mctdhf_orbitals, num_spin_orbitals, num_mctdhf_slater_dets)
    # Normalize every step during imaginary time prop
    #C = np.divide(C, np.sqrt(C.T.conj()@C))

    # Might be better to move this renormalization to inside the mctdhf_func?
    b = np.divide(b, np.sqrt(np.einsum('ij, ij -> j', b.conj(), b)))
    b = gram_schmidt(b)
    bc = b.conj()



    ## Test to make sure b_dot is orthogonal to b (otherwise orthonormality will be lost.)  
    h_1 = np.einsum('jm, ij -> im', b, h)
    h_2 = np.einsum('in, im -> nm', bc, h_1)
    # h_3 = np.einsum('in, nm -> im', b, h_2)
    # b_dot = -(h_1-h_3)
    # print(f'b@b_dot = \n{b.T.conj()@b_dot}')

    g_2 = np.einsum('jq, ls, ijkl -> iqks', bc, b, g)
    g_3 = np.einsum('kr, iqks -> iqrs', b, g_2)
    g_4 = np.einsum('ip, iqrs -> pqrs', bc, g_3)
    #g_5 = np.einsum('ip, pqrs -> iqrs', b, g_4)

    H = SlaterCondonHamiltonian(num_mctdhf_orbitals, num_electrons, h_2, g_4).get_hamiltonian()

    Es.append(np.real(C.conj().T@H@C))
    Cs.append(C)
    bs.append(b)

    solver.y = Cb_to_y(C, b)
else:
    if solver.status != 'finished':
        raise(RuntimeError(msg))

In [ ]:
C_f, b_f = y_to_Cb(solver.y, num_mctdhf_orbitals, num_spin_orbitals, num_mctdhf_slater_dets)
#print(f"Final b =\n{b_f}")
#print(f"Final C =\n{C_f}")
print(f"Energy: {Es[-1]}")
print(f"Energy deviation =\n{Es[-1]-E_exact}")

#print(f"Overlap matrix for final b:\n{b_f.T@b_f}")

print(f"Max overlap coeff (ideally zero): {np.abs(np.max(b_f.T@b_f-np.eye(num_mctdhf_orbitals)))}")


b_arr = np.array(bs)
Es = np.array(Es)
Cs = np.array(Cs)
t = np.linspace(t_init, t_final, len(Cs))

fig, ax = plt.subplots(1, 3, figsize=(16,6))
ax[0].plot(t, np.real(Cs))
ax[0].set_title('Cs')
ax[1].plot(t, np.real(b_arr.reshape(b_arr.shape[0],*b_init.shape)[:,:,0]))
ax[1].set_title('bs')
ax[2].plot(t, E_exact*np.ones_like(Es), 'k--')
ax[2].plot(t, Es)
ax[2].set_title('E')
plt.show()

In [ ]:
D, _ = get_RDMs(num_mctdhf_orbitals, num_electrons, C_f)

In [ ]:
plt.plot(x, pot(x))
plt.plot(x, np.diag(spf.T@(b_f@D@b_f.T)@spf)*10)
plt.show()

In [ ]:
class MCDTHF:
    def __init__(self, h, g, num_mctdhf_orbitals, num_spin_orbitals, num_electrons):
        self.h = h
        self.g = g
        self.num_mctdhf_orbitals = num_mctdhf_orbitals
        self.num_spin_orbitals = num_spin_orbitals
        self.num_electrons = num_electrons
        self.num_slater_dets = int(comb(num_mctdhf_orbitals, num_electrons))
        self.I = np.eye(self.num_slater_dets)
    
    def __call__(self, t, y):
        
        C, b = y_to_Cb(y, self.num_mctdhf_orbitals, self.num_spin_orbitals, self.num_slater_dets)
        bc = b.conj()

        ## These two implementations are identical. Not sure which is better.
        h_1 = np.einsum('jm, ij -> im', b, self.h)
        h_2 = np.einsum('in, im -> nm', bc, h_1)
        h_3 = np.einsum('in, nm -> im', b, h_2)
        # h_1 = self.h@b
        # h_2 = bc.T@h_1
        # h_3 = b@h_2

        g_2 = np.einsum('jq, ls, ijkl -> iqks', bc, b, self.g)
        g_3 = np.einsum('kr, iqks -> iqrs', b, g_2)
        g_4 = np.einsum('ip, iqrs -> pqrs', bc, g_3)
        g_5 = np.einsum('ip, pqrs -> iqrs', b, g_4)

        
        D, d = get_RDMs(self.num_mctdhf_orbitals, self.num_electrons, C)
        D_inv = np.linalg.pinv(D)

        b_dot = -1j*(h_1 - h_3 + np.einsum('np, pqrs, iqrs -> in', D_inv, d, g_3-g_5))


        H = SlaterCondonHamiltonian(self.num_mctdhf_orbitals, self.num_electrons, h_2, g_4).get_hamiltonian()

        ## From Beck paper, replacing HC with (H-IE)C should keep the wave function normalized (should be able to remove renormalization further down)
        E = C.conj().T@H@C/(C.conj().T@C)
        C_dot = -1j*(H-self.I*E)@C 
    
        ## Regular way
        #C_dot = -1j*H@C


        return Cb_to_y(C_dot, b_dot)

In [ ]:
mctdhf_fun = MCDTHF(h, g, num_mctdhf_orbitals, num_spin_orbitals, num_electrons)

y_init = Cb_to_y(C_f, b_f)

Cs_rt = [C_f]
bs_rt = [b_f]

Es_rt = [Es[-1]]

t_init = 0
t_final = 5.0
solver = DOP853(mctdhf_fun, t_init, y_init, t_final, max_step = 1e-2)
#solver = RK45(mctdhf_fun, t_init, y_init, t_final, max_step = 5e-3)
#solver = RK23(mctdhf_fun, t_init, y_init, t_final, max_step = 0.001)

while solver.status == 'running':
    msg = solver.step()

    C, b = y_to_Cb(solver.y, num_mctdhf_orbitals, num_spin_orbitals, num_mctdhf_slater_dets)
    # Normalize every step during imaginary time prop
    #C = np.divide(C, np.sqrt(C.T.conj()@C))

    # Might be better to move this renormalization to inside the mctdhf_func?
    # b = np.divide(b, np.sqrt(np.einsum('ij, ij -> j', b.conj(), b)))
    # b = gram_schmidt(b)
    # bc = b.conj()


    ## Test to make sure b_dot is orthogonal to b (otherwise orthonormality will be lost.)  
    h_1 = np.einsum('jm, ij -> im', b, h)
    h_2 = np.einsum('in, im -> nm', bc, h_1)
    # h_3 = np.einsum('in, nm -> im', b, h_2)
    # b_dot = -(h_1-h_3)
    # print(f'b@b_dot = \n{b.T.conj()@b_dot}')

    g_2 = np.einsum('jq, ls, ijkl -> iqks', bc, b, g)
    g_3 = np.einsum('kr, iqks -> iqrs', b, g_2)
    g_4 = np.einsum('ip, iqrs -> pqrs', bc, g_3)
    #g_5 = np.einsum('ip, pqrs -> iqrs', b, g_4)

    H = SlaterCondonHamiltonian(num_mctdhf_orbitals, num_electrons, h_2, g_4).get_hamiltonian()

    #D, d = get_RDMs(num_mctdhf_orbitals, num_electrons, C)

    #Es = np.append(Es, np.real(np.einsum('pq, pq', D, h_2)))
    Es_rt.append(np.real(C.conj().T@H@C))
    Cs_rt.append(C)
    bs_rt.append(b)

    solver.y = Cb_to_y(C, b)
else:
    if solver.status != 'finished':
        raise(RuntimeError(msg))

In [ ]:
Es_rt = np.array(Es_rt)
Cs_rt = np.array(Cs_rt)
b_arr = np.array(bs_rt)

t = np.linspace(t_init, t_final, len(Cs_rt))

fig, ax = plt.subplots(1, 3, figsize=(16,6))
ax[0].plot(t, np.real(Cs_rt))
ax[0].set_title('Cs')
ax[1].plot(t, np.real(b_arr.reshape(b_arr.shape[0],*b_init.shape)[:,:,0]))
ax[1].set_title('bs')
ax[2].plot(t, E_exact*np.ones_like(Es_rt), 'k--')
ax[2].plot(t, Es_rt)
ax[2].set_title('E')
plt.show()